# Anomaly Detection on QLD Government Open Data

**Data Source:** TMR New Business Registration Transactions (data.qld.gov.au)  
**Records:** 304,553 transactions across 654 suburbs, 12 months  
**Snowflake Features:** `SNOWFLAKE.ML.ANOMALY_DETECTION`, `CORTEX.COMPLETE`, Notebooks

In [ ]:
import snowflake.connector
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME') or 'AU_DEMO50')
cur = conn.cursor()
cur.execute('USE DATABASE CDSB_DEMO')
cur.execute('USE SCHEMA RAW')
cur.execute('USE WAREHOUSE COMPUTE_WH')
print('Connected to CDSB_DEMO.RAW')

## 1. Explore the Data
Real QLD Government open data — 304K vehicle registration transactions from TMR

In [ ]:
df_raw = pd.read_sql("""
    SELECT MONTH, SUBURB, POSTCODE, TRANSACTION_TYPE, MAKE_NAME, MODEL_NAME,
           VALID_HANDLED_TRANSACTIONS
    FROM TMR_REGISTRATIONS
    LIMIT 10
""", conn)
df_raw

In [ ]:
df_monthly = pd.read_sql("SELECT * FROM TMR_MONTHLY_REGISTRATIONS", conn)

fig = px.bar(df_monthly, x='MONTH', y='TOTAL_REGISTRATIONS',
             title='QLD Monthly New Business Vehicle Registrations',
             labels={'TOTAL_REGISTRATIONS': 'Registrations', 'MONTH': 'Month'})
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
df_regional = pd.read_sql("SELECT * FROM TMR_REGIONAL_REGISTRATIONS", conn)

fig = px.line(df_regional, x='MONTH', y='TOTAL_REGISTRATIONS', color='REGION',
              title='Registrations by QLD Region',
              labels={'TOTAL_REGISTRATIONS': 'Registrations', 'MONTH': 'Month'})
fig.update_layout(template='plotly_white')
fig.show()

## 2. Train Anomaly Detection Model

One SQL statement — Snowflake handles feature engineering, model selection, and training.  
We use **multi-series** mode: train one model across all 654 suburbs simultaneously.

In [ ]:
cur.execute("""
    CREATE OR REPLACE VIEW TMR_TRAINING_DATA AS
    SELECT TS, SERIES_ID, REGISTRATIONS
    FROM TMR_SUBURB_MONTHLY
    WHERE TS < '2026-02-01'
      AND SERIES_ID IN (
        SELECT SERIES_ID FROM TMR_SUBURB_MONTHLY
        WHERE TS < '2026-02-01'
        GROUP BY SERIES_ID HAVING COUNT(DISTINCT TS) >= 8
      )
""")

cur.execute("""
    CREATE OR REPLACE VIEW TMR_TEST_DATA AS
    SELECT TS, SERIES_ID, REGISTRATIONS
    FROM TMR_SUBURB_MONTHLY
    WHERE TS >= '2026-02-01'
      AND SERIES_ID IN (
        SELECT SERIES_ID FROM TMR_SUBURB_MONTHLY
        WHERE TS < '2026-02-01'
        GROUP BY SERIES_ID HAVING COUNT(DISTINCT TS) >= 8
      )
""")

train_count = pd.read_sql("SELECT COUNT(*) as n FROM TMR_TRAINING_DATA", conn).iloc[0,0]
test_count = pd.read_sql("SELECT COUNT(*) as n FROM TMR_TEST_DATA", conn).iloc[0,0]
print(f'Training: {train_count} rows (Apr 2025 - Jan 2026)')
print(f'Test: {test_count} rows (Feb - Mar 2026)')

In [ ]:
%%time
cur.execute("""
    CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION tmr_anomaly_model(
        INPUT_DATA => TABLE(TMR_TRAINING_DATA),
        SERIES_COLNAME => 'SERIES_ID',
        TIMESTAMP_COLNAME => 'TS',
        TARGET_COLNAME => 'REGISTRATIONS',
        LABEL_COLNAME => '',
        CONFIG_OBJECT => {'on_error': 'skip'}
    )
""")
print('Model trained!', cur.fetchone())

## 3. Detect Anomalies
Run the model on recent data to find unusual registration patterns

In [ ]:
cur.execute("""
    CREATE OR REPLACE TABLE TMR_ANOMALY_RESULTS AS
    SELECT *
    FROM TABLE(
        tmr_anomaly_model!DETECT_ANOMALIES(
            INPUT_DATA => TABLE(TMR_TEST_DATA),
            SERIES_COLNAME => 'SERIES_ID',
            TIMESTAMP_COLNAME => 'TS',
            TARGET_COLNAME => 'REGISTRATIONS',
            CONFIG_OBJECT => {'prediction_interval': 0.95}
        )
    )
""")

df_anomalies = pd.read_sql("""
    SELECT * FROM TMR_ANOMALY_RESULTS
    ORDER BY IS_ANOMALY DESC, PERCENTILE DESC
""", conn)

n_anomalies = df_anomalies['IS_ANOMALY'].sum()
print(f'Detected {n_anomalies} anomalies out of {len(df_anomalies)} suburb-months')
df_anomalies.head(20)

In [ ]:
df_anom_only = df_anomalies[df_anomalies['IS_ANOMALY'] == True].copy()
df_anom_only = df_anom_only.sort_values('PERCENTILE', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_anom_only['SERIES'].head(25),
    y=df_anom_only['Y'].head(25),
    name='Actual', marker_color='red'
))
fig.add_trace(go.Bar(
    x=df_anom_only['SERIES'].head(25),
    y=df_anom_only['FORECAST'].head(25),
    name='Expected', marker_color='lightblue'
))
fig.update_layout(
    title='Top 25 Anomalous Suburbs: Actual vs Expected Registrations',
    barmode='group', template='plotly_white', xaxis_tickangle=-45
)
fig.show()

## 4. Time Series Deep Dive — Single Suburb
Pick the most anomalous suburb and visualise its full history with detection bounds

In [ ]:
target_suburb = df_anom_only.iloc[0]['SERIES'] if len(df_anom_only) > 0 else 'BRISBANE CITY'

df_suburb = pd.read_sql(f"""
    SELECT s.TS, s.REGISTRATIONS, a.FORECAST, a.LOWER_BOUND, a.UPPER_BOUND, a.IS_ANOMALY
    FROM TMR_SUBURB_MONTHLY s
    LEFT JOIN TMR_ANOMALY_RESULTS a ON s.TS = a.TS AND s.SERIES_ID = a.SERIES
    WHERE s.SERIES_ID = '{target_suburb}'
    ORDER BY s.TS
""", conn)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['REGISTRATIONS'],
                         mode='lines+markers', name='Actual', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['FORECAST'],
                         mode='lines', name='Forecast', line=dict(color='green', dash='dash')))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['UPPER_BOUND'],
                         mode='lines', name='Upper Bound', line=dict(color='grey', dash='dot')))
fig.add_trace(go.Scatter(x=df_suburb['TS'], y=df_suburb['LOWER_BOUND'],
                         mode='lines', name='Lower Bound', line=dict(color='grey', dash='dot'),
                         fill='tonexty', fillcolor='rgba(200,200,200,0.2)'))

anomaly_pts = df_suburb[df_suburb['IS_ANOMALY'] == True]
if len(anomaly_pts) > 0:
    fig.add_trace(go.Scatter(x=anomaly_pts['TS'], y=anomaly_pts['REGISTRATIONS'],
                             mode='markers', name='ANOMALY',
                             marker=dict(color='red', size=14, symbol='x')))

fig.update_layout(title=f'Registration Anomalies — {target_suburb}', template='plotly_white')
fig.show()

## 5. LLM-Powered Anomaly Interpretation
Use `CORTEX.COMPLETE` to have an AI explain the anomalies in plain English — no data science expertise needed

In [ ]:
df_top = pd.read_sql("""
    SELECT SERIES as SUBURB, TS as MONTH, Y as ACTUAL,
           ROUND(FORECAST, 0) as EXPECTED,
           ROUND(Y - FORECAST, 0) as DEVIATION,
           ROUND(PERCENTILE, 4) as CONFIDENCE
    FROM TMR_ANOMALY_RESULTS
    WHERE IS_ANOMALY = TRUE
    ORDER BY ABS(Y - FORECAST) DESC
    LIMIT 15
""", conn)
print(df_top.to_string(index=False))

In [ ]:
anomaly_text = df_top.to_string(index=False)

llm_result = pd.read_sql("""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        'You are a government data analyst for Queensland Transport and Main Roads.
Below are detected anomalies in new business vehicle registration transactions across QLD suburbs.
Each row shows suburb, month, actual registrations, expected registrations, and deviation.

ANOMALY DATA:
""" + anomaly_text.replace("'", "''") + """

Please:
1. Summarise the top patterns (which suburbs, direction, significance)
2. Suggest possible real-world explanations (new developments, seasonal effects, economic changes)
3. Recommend which anomalies need urgent investigation
4. Suggest additional data sources that would help confirm these patterns'
    ) as ANALYSIS
""", conn)

print(llm_result.iloc[0]['ANALYSIS'])

## 6. Automate with Tasks & Alerts
In production: retrain monthly, alert on new anomalies — all SQL-native

In [ ]:
print("""
-- PRODUCTION AUTOMATION (not executing in demo)

-- Retrain model on 1st of each month
CREATE OR REPLACE TASK tmr_anomaly_retrain
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 6 1 * * Australia/Brisbane'
AS
  CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION tmr_anomaly_model(
    INPUT_DATA => TABLE(TMR_TRAINING_DATA),
    SERIES_COLNAME => 'SERIES_ID',
    TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'REGISTRATIONS'
  );

-- Weekly alert for new anomalies
CREATE OR REPLACE ALERT tmr_anomaly_alert
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 7 * * MON Australia/Brisbane'
  IF (EXISTS (
    SELECT 1 FROM TABLE(tmr_anomaly_model!DETECT_ANOMALIES(
      INPUT_DATA => TABLE(TMR_TEST_DATA),
      SERIES_COLNAME => 'SERIES_ID', TIMESTAMP_COLNAME => 'TS',
      TARGET_COLNAME => 'REGISTRATIONS'
    )) WHERE IS_ANOMALY = TRUE
  ))
  THEN CALL SYSTEM$SEND_EMAIL(
    'tmr_notifications', 'analyst@tmr.qld.gov.au',
    'TMR Registration Anomaly Detected',
    'New anomalies detected in vehicle registration data. Please review.'
  );
""")

## Key Takeaways

| Step | Snowflake Feature | Effort |
|------|------------------|--------|
| Load open data | Stage + COPY INTO | 2 min |
| Train model | `SNOWFLAKE.ML.ANOMALY_DETECTION` | 1 SQL statement |
| Detect anomalies | `model!DETECT_ANOMALIES()` | 1 SQL statement |
| Explain with AI | `CORTEX.COMPLETE` | 1 SQL statement |
| Automate | Tasks + Alerts | Standard Snowflake |

**Total: from raw data to AI-powered anomaly alerts in under 30 minutes**